# 세션 1 — RAG 개요와 평가셋

### RAG가 하는 일
```
질문 → [검색] 관련 내용 찾기 → [LLM] 찾은 내용으로 답하기 → 답변
```
LLM은 우리 문서를 모릅니다. 그래서 문서에서 관련 부분을 찾아 LLM에게 같이 넘겨 답하게 하는 방식이 RAG입니다.

### 평가셋을 먼저 만드는 이유
추출·검색·재정렬을 바꿀 때마다 좋아졌는지 숫자로 확인하려면 정답이 정해진 문제집이 필요합니다.
이번 세션에서 그 12문항을 만듭니다.

In [7]:
# 준비 셀 — 경로와 API 키를 읽어 온다 (세션 0과 동일)
import os, time, pickle
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()                            # .env 의 OPENAI_API_KEY 를 환경변수로 등록

ROOT = Path.cwd().parent                 # notebooks/ 의 상위 = 프로젝트 폴더
PDF = ROOT / "data" / "저출생_반전을_위한_대책_관계부처_합동_.pdf"
CACHE = ROOT / "cache"
CACHE.mkdir(exist_ok=True)

print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")), "| PDF:", PDF.exists())

OpenAI key: True | PDF: True


## 1-1. 데이터 — 저출생 추세 반전을 위한 대책 (2024.6.19)

정책 설명 글에 현행→개선 비교표가 많이 섞인 문서입니다. 표가 많아 *추출 방식*이 중요하고,
"늘봄학교·신생아 특례대출" 같은 고유명사가 많아 *키워드 검색*도 잘 듣습니다(다음 세션들 주제).

In [8]:
# PDF 를 열어 어떤 문서인지 눈으로 확인한다
import fitz                              # PyMuPDF — PDF 를 다루는 라이브러리 (이름이 fitz)

doc = fitz.open(PDF)
print("페이지 수:", doc.page_count)
print(doc[0].get_text()[:250])           # 첫 페이지 텍스트 앞 250자만 미리보기
doc.close()                              # 파일은 다 쓰면 닫아 준다

페이지 수: 40
저출생 추세 반전을 위한 대책
2024. 6. 19.
저출산고령사회위원회
관계부처 합동
3대 분야 15대 핵심 과제
1. 일·가정 양립
➊ 단기 육아휴직 도입(年1회 2주 단위 육아휴직 허용)18페이지
➋ 육아휴직 급여 최대 상한 인상(現 150→최대 250만원)19페이지 및 
육아휴직 대체인력지원금 신설 및 지원금 확대(現 80→120만원)23페이
지
➌ 아빠 출산휴가 기간(現 10일→20일, 근무일 기준), 청구기한(現90→
120일) 및 


## 1-2. 평가셋 12문항

조건은 두 가지입니다. 정답이 문서에 분명히 있을 것, 그리고 표기가 조금 달라도 맞다고 인정할 것
(예: "250만원" = "월 250만원").

In [9]:
# 정답이 문서에서 확인된 평가 문항 12개
# q = 질문, a = 정답. 이 12문항으로 오늘 하루 종일 점수를 매긴다.
EVAL = [
    {"q": "2023년 한국 합계출산율은?",                         "a": "0.72명"},
    {"q": "2023년 출생아 수는?",                               "a": "23만명"},
    {"q": "정부가 2030년까지 목표로 하는 합계출산율은?",        "a": "1.0"},
    {"q": "육아휴직 급여 최대 상한은 얼마로 인상되나?",         "a": "250만원"},
    {"q": "아빠(배우자) 출산휴가 기간은 며칠로 확대되나?",      "a": "20일"},
    {"q": "부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?", "a": "1년 6개월"},
    {"q": "대체인력지원금은 월 얼마로 인상되나?",              "a": "120만원"},
    {"q": "임기 내 공공보육 이용률 목표는?",                   "a": "50%"},
    {"q": "0세반 교사 대 영아 비율은 어떻게 개선되나?",        "a": "1:2"},
    {"q": "'25년 이후 출산가구 신생아 특례대출 소득요건은?",    "a": "2.5억원"},
    {"q": "출산가구 대상 주택공급은 연 몇 호로 확대되나?",      "a": "12만호"},
    {"q": "보고서가 제시한 한국의 육아휴직 소득대체율은?",      "a": "38.6%"},
]

import re

def norm(s):
    # 채점 전에 문자열을 "표준형"으로 만든다 → 표기 차이를 흡수
    # 예: "월 250만원" → "250만원",  "20근무일" → "20일"
    s = s.replace(" ", "").replace(",", "").replace("근무일", "일")
    return s[1:] if s.startswith("월") else s      # 맨 앞의 "월"만 떼어 낸다

def is_correct(answer, expected):
    # LLM 답변(answer) 안에 정답(expected)이 들어 있으면 맞은 것으로 채점한다
    a, e = norm(answer), norm(expected)
    # "1.0" 같은 숫자로 끝나는 정답은 "1.06" 안에서도 발견되므로,
    # 매칭 바로 뒤에 숫자가 이어지면 오탐으로 보고 제외한다
    for m in re.finditer(re.escape(e), a):        # 답변 속 정답 위치를 전부 찾아서
        if e[-1].isdigit() and m.end() < len(a) and a[m.end()].isdigit():
            continue                              # 뒤에 숫자가 붙은 경우("1.06")는 무시
        return True
    return False

In [10]:
EVAL

[{'q': '2023년 한국 합계출산율은?', 'a': '0.72명'},
 {'q': '2023년 출생아 수는?', 'a': '23만명'},
 {'q': '정부가 2030년까지 목표로 하는 합계출산율은?', 'a': '1.0'},
 {'q': '육아휴직 급여 최대 상한은 얼마로 인상되나?', 'a': '250만원'},
 {'q': '아빠(배우자) 출산휴가 기간은 며칠로 확대되나?', 'a': '20일'},
 {'q': '부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?', 'a': '1년 6개월'},
 {'q': '대체인력지원금은 월 얼마로 인상되나?', 'a': '120만원'},
 {'q': '임기 내 공공보육 이용률 목표는?', 'a': '50%'},
 {'q': '0세반 교사 대 영아 비율은 어떻게 개선되나?', 'a': '1:2'},
 {'q': "'25년 이후 출산가구 신생아 특례대출 소득요건은?", 'a': '2.5억원'},
 {'q': '출산가구 대상 주택공급은 연 몇 호로 확대되나?', 'a': '12만호'},
 {'q': '보고서가 제시한 한국의 육아휴직 소득대체율은?', 'a': '38.6%'}]

In [11]:
# 12문항을 한눈에 훑어본다 (질문과 정답)
for i, item in enumerate(EVAL, 1):
    print(f"{i:2}. {item['q']}  ({item['a']})")

 1. 2023년 한국 합계출산율은?  (0.72명)
 2. 2023년 출생아 수는?  (23만명)
 3. 정부가 2030년까지 목표로 하는 합계출산율은?  (1.0)
 4. 육아휴직 급여 최대 상한은 얼마로 인상되나?  (250만원)
 5. 아빠(배우자) 출산휴가 기간은 며칠로 확대되나?  (20일)
 6. 부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?  (1년 6개월)
 7. 대체인력지원금은 월 얼마로 인상되나?  (120만원)
 8. 임기 내 공공보육 이용률 목표는?  (50%)
 9. 0세반 교사 대 영아 비율은 어떻게 개선되나?  (1:2)
10. '25년 이후 출산가구 신생아 특례대출 소득요건은?  (2.5억원)
11. 출산가구 대상 주택공급은 연 몇 호로 확대되나?  (12만호)
12. 보고서가 제시한 한국의 육아휴직 소득대체율은?  (38.6%)


## 1-3. 정답이 정말 문서에 있는지 확인

평가셋을 믿으려면 각 정답이 PDF 본문에 실제로 있어야 합니다. 12개 모두 확인되어야 합니다.

In [12]:
# PDF 전체 텍스트를 뽑아, 12개 정답이 실제로 본문에 있는지 확인한다
doc = fitz.open(PDF)
full_text = "\n".join(page.get_text() for page in doc)   # 40쪽 텍스트를 한 덩어리로
doc.close()

# norm 은 1-2에서 정의한 것을 그대로 쓴다 (표기 차이를 흡수해서 찾기 위해)
found = sum(item["a"] in full_text or norm(item["a"]) in norm(full_text) for item in EVAL)
for item in EVAL:
    ok = item["a"] in full_text or norm(item["a"]) in norm(full_text)
    print(("O" if ok else "X"), item["a"])
print(f"\n문서에서 확인된 정답: {found} / {len(EVAL)}")

O 0.72명
O 23만명
O 1.0
O 250만원
O 20일
O 1년 6개월
O 120만원
O 50%
O 1:2
O 2.5억원
O 12만호
O 38.6%

문서에서 확인된 정답: 12 / 12


## 1-4. 채점 — 표기 차이 흡수

LLM은 같은 사실도 다르게 말합니다("월 250만원", "최대 250만 원"). `is_correct` 가 이런 차이를 흡수합니다.

In [13]:
# is_correct 가 표기 차이를 잘 흡수하는지 세 가지 답안으로 시험해 본다
samples = [
    ("육아휴직 급여는 월 250만원으로 오릅니다.", "250만원"),   # "월"이 붙어도 O
    ("아빠 출산휴가는 20근무일입니다.", "20일"),               # "근무일"이라 써도 O
    ("정보를 찾을 수 없습니다.", "38.6%"),                     # 정답이 없으면 X
]
for answer, expected in samples:
    print(("O" if is_correct(answer, expected) else "X"), f"{expected}  <-  {answer}")

O 250만원  <-  육아휴직 급여는 월 250만원으로 오릅니다.
O 20일  <-  아빠 출산휴가는 20근무일입니다.
X 38.6%  <-  정보를 찾을 수 없습니다.


## 1-5. 왜 RAG가 필요한가 — 베이스라인 점수

평가셋이 생겼으니, **아무 기법도 안 쓴 출발점**을 실제로 재봅니다. 두 극단을 비교합니다.

1. **LLM 단독** — 문서를 주지 않고 질문만 던집니다. 모델이 모르는 최신·세부 수치는 틀리거나 지어냅니다(환각).
2. **전체 문서 통째** — 40페이지 전부를 LLM에 같이 넣습니다. 잘 맞지만, **매 질문마다 문서 전체를 보내** 느리고 비싸며, 문서가 커지면 아예 불가능합니다.

→ 그래서 **필요한 부분만 찾아서** 주는 RAG가 필요합니다(다음 세션부터). 여기서 나온 점수가 앞으로 끌어올릴 출발선입니다.

In [14]:
# 베이스라인 두 가지를 준비한다 — "검색 없이" 답하는 두 극단
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def ask_no_doc(q):                       # 1) 문서 없이 LLM에게만 물어보기
    return llm.invoke(q).content

# {context} 와 {question} 자리에 문서와 질문이 채워지는 프롬프트 틀
BASELINE_PROMPT = """다음 문서를 바탕으로 질문에 답하세요. 문서에 없으면 모른다고 하세요.

문서:
{context}

질문: {question}
답변:"""

def ask_full_doc(q):                     # 2) full_text(1-3에서 추출)를 통째로 맥락으로
    return llm.invoke(BASELINE_PROMPT.format(context=full_text, question=q)).content

In [15]:
# 한 문항으로 차이를 직접 본다 (정답: 250만원)
q = "육아휴직 급여 최대 상한은 얼마로 인상되나?"
print("LLM 단독  :", ask_no_doc(q)[:80])      # 문서 없이 → 지어내거나 모른다고 한다
print("전체 문서 :", ask_full_doc(q)[:80])    # 문서 통째 → 대부분 정확히 답한다

LLM 단독  : 2023년 기준으로 한국의 육아휴직 급여는 최대 상한이 월 250만 원으로 인상되었습니다. 이는 육아휴직을 사용하는 부모가 받을 수 있는 최대 
전체 문서 : 육아휴직 급여 최대 상한은 250만원으로 인상됩니다.


In [16]:
# 채점 헬퍼 — 12문항을 전부 풀려 점수를 매긴다 (베이스라인용)
def score(ask_fn, label):
    t0 = time.time(); correct = 0
    for item in EVAL:                    # 12문항을 하나씩
        ans = ask_fn(item["q"])          # 질문을 던지고
        ok = is_correct(ans, item["a"]); correct += ok    # 채점해서 누적
        print(("O" if ok else "X"), item["a"], "->", ans[:40].replace("\n", " "))
    print(f"\n[{label}] {correct}/{len(EVAL)}  ({time.time()-t0:.0f}s)")
    return correct

score(ask_no_doc, "LLM 단독")            # 12콜 — 1분 안팎

X 0.72명 -> 2023년 한국의 합계출산율에 대한 구체적인 수치는 제가 제공할 수 없습
X 23만명 -> 2023년 출생아 수에 대한 구체적인 통계는 아직 제공되지 않았습니다. 
X 1.0 -> 한국 정부는 2030년까지 합계출산율을 1.6명으로 목표하고 있습니다. 
O 250만원 -> 2023년 기준으로 한국의 육아휴직 급여는 최대 상한이 월 250만 원으
X 20일 -> 아빠(배우자)의 출산휴가는 국가나 지역에 따라 다르게 규정되어 있습니다.
X 1년 6개월 -> 부모가 모두 3개월 이상 육아휴직을 사용할 경우, 총 육아휴직 기간은 부
X 120만원 -> 대체인력지원금의 인상 여부와 구체적인 금액은 정부의 정책에 따라 달라질 
X 50% -> 2023년 기준으로 한국 정부의 공공보육 이용률 목표는 2025년까지 4
X 1:2 -> 0세반 교사 대 영아 비율을 개선하기 위해서는 여러 가지 접근 방법이 있
X 2.5억원 -> 2025년 이후 출산가구 신생아 특례대출의 소득 요건에 대한 구체적인 정
X 12만호 -> 출산가구 대상 주택공급 확대에 대한 구체적인 내용은 정부의 정책에 따라 
X 38.6% -> 한국의 육아휴직 소득대체율은 2023년 기준으로 2022년 7월부터 시행

[LLM 단독] 1/12  (30s)


1

In [17]:
# 전체 문서를 통째로 넣는 방식 — 잘 맞지만 매 질문마다 5만 자를 보낸다
print(f"문서 길이: {len(full_text):,}자 — 매 질문마다 이 전체를 LLM에 보낸다")
score(ask_full_doc, "전체 문서 통째")     # 12콜 — 1분 안팎

문서 길이: 54,724자 — 매 질문마다 이 전체를 LLM에 보낸다
O 0.72명 -> 2023년 한국의 합계출산율은 0.72명입니다.
X 23만명 -> 문서에 해당 정보가 없습니다.
O 1.0 -> 2030년까지 목표로 하는 합계출산율은 1.0입니다.
O 250만원 -> 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.
O 20일 -> 아빠(배우자) 출산휴가 기간은 10일에서 20일로 확대됩니다.
O 1년 6개월 -> 부모가 모두 3개월 이상 육아휴직을 사용할 경우 총 육아휴직 기간은 1년
O 120만원 -> 대체인력지원금은 월 120만원으로 인상됩니다.
O 50% -> 임기 내 공공보육 이용률 목표는 50%로 확대하는 것입니다.
O 1:2 -> 0세반 교사 대 영아 비율은 1:3에서 1:2로 개선될 예정입니다.
O 2.5억원 -> '25년 이후 출산가구 신생아 특례대출 소득요건은 2.5억원으로 완화됩니
O 12만호 -> 출산가구 대상 주택공급은 연 12만 호로 확대됩니다.
X 38.6% -> 보고서에 따르면 한국의 육아휴직 소득대체율은 현재 80%이며, 최대 15

[전체 문서 통째] 10/12  (74s)


10

## 직접 해보기

> 예시 코드를 **새 셀에 복사해** 실행해 보세요.

**1. 다른 평가셋 문항으로 LLM 단독 vs 전체 문서 답변을 비교해 보세요.**
```python
q = EVAL[6]["q"]                      # "대체인력지원금은 월 얼마로 인상되나?"  (숫자를 바꿔 다른 문항도)
print("LLM 단독  :", ask_no_doc(q)[:80])
print("전체 문서 :", ask_full_doc(q)[:80])
```

**2. 표기를 바꾼 답안을 만들어 `is_correct` 가 O로 채점하는지 확인해 보세요.**
```python
print(is_correct("배우자 출산휴가는 20 근무일로 늘어납니다.", "20일"))    # True가 나와야 정상
print(is_correct("급여 상한은 월 250만 원입니다.", "250만원"))            # True가 나와야 정상
print(is_correct("현행 150만원이 유지됩니다.", "250만원"))                # False가 나와야 정상
```

**3. 아래 심화에서 13번째 문항을 직접 만들어 보세요.** (완성 예시와 빈 템플릿이 준비되어 있습니다)

In [18]:
q = EVAL[6]["q"]                      # "대체인력지원금은 월 얼마로 인상되나?"  (숫자를 바꿔 다른 문항도)
print("LLM 단독  :", ask_no_doc(q)[:80])
print("전체 문서 :", ask_full_doc(q)[:80])

LLM 단독  : 대체인력지원금의 인상 여부와 구체적인 금액은 정부의 정책에 따라 달라질 수 있습니다. 2023년의 경우, 대체인력지원금에 대한 구체적인 인상 내
전체 문서 : 대체인력지원금은 월 120만원으로 인상됩니다.


In [21]:
print(is_correct("배우자 출산휴가는 20 근무일로 늘어납니다.", "20일"))    # True가 나와야 정상
print(is_correct("급여 상한은 월 250만 원입니다.", "250만원"))            # True가 나와야 정상
print(is_correct("현행 150만원이 유지됩니다.", "250만원"))                # False가 나와야 정상

True
True
False


## 시간이 남으면 (심화) — 13번째 문항 직접 만들기

좋은 평가 문항인지 확인하는 3단계 점검을 그대로 따라 해 봅니다.

| 점검 | 내용 | 실패하면 |
|------|------|---------|
| ① 정답이 문서에 있는가 | 본문에서 정답 문자열이 실제로 확인돼야 합니다 | 문항이 아니라 평가셋 결함 |
| ② 정답이 하나로 확정되는가 | 헤드라인과 본문이 다르게 읽히면(2억 vs 2.5억) 질문에 조건을 박습니다 | 모호 문항 → 채점 불가 |
| ③ 표기 변형이 흡수되는가 | "월 250만원"·"20근무일" 같은 변형이 `is_correct` 로 O 처리돼야 합니다 | 정답인데 X로 채점 |

아래는 완성 예시 — 1쪽 목차의 "아빠 출산휴가 청구기한 (現90→120일)"에서 만든 문항입니다.

In [22]:
# 완성 예시 — 새 문항을 3단계로 점검한다
new_item = {"q": "아빠 출산휴가 청구기한은 며칠로 확대되나?", "a": "120일"}

# ① 정답이 문서에 있는가 (없으면 문항이 아니라 평가셋 결함)
in_doc = new_item["a"] in full_text or norm(new_item["a"]) in norm(full_text)
print("① 문서에서 확인:", "O" if in_doc else "X")

# ② 정답이 하나로 확정되는가 — 전체 문서 베이스라인에게 물어 확인
ans = ask_full_doc(new_item["q"])
print("② 모델 답변    :", ans[:60])

# ③ 표기 변형까지 흡수해 채점되는가
print("③ 채점         :", "O" if is_correct(ans, new_item["a"]) else "X")

① 문서에서 확인: O
② 모델 답변    : 아빠 출산휴가 청구기한은 90일에서 120일로 확대됩니다.
③ 채점         : O


In [23]:
# 여러분 차례 — 문서에서 사실 하나를 골라 직접 문항을 만들어 보세요.
# 힌트: 1쪽 목차(15대 과제)에서 "現 A → B" 패턴을 찾으면 만들기 쉽다.
my_item = {"q": "2030년도에 부과방식이용률은?", "a": "9.2%"}   # TODO: 질문과 정답 채우기

if my_item["q"] and my_item["a"]:                 # q 와 a 를 채웠을 때만 점검 실행
    in_doc = my_item["a"] in full_text or norm(my_item["a"]) in norm(full_text)
    print("① 문서에서 확인:", "O" if in_doc else "X")
    ans = ask_full_doc(my_item["q"])
    print("② 모델 답변    :", ans[:60])
    print("③ 채점         :", "O" if is_correct(ans, my_item["a"]) else "X")
else:
    print("my_item 의 q / a 를 채운 뒤 다시 실행하세요.")

① 문서에서 확인: O
② 모델 답변    : 문서에 해당 정보가 없습니다.
③ 채점         : X


## 정리
- RAG는 검색으로 관련 문서를 찾아 LLM에 같이 줘서 답하게 하는 것입니다.
- 평가셋 12문항은 정답이 문서에서 모두 확인됐습니다. 앞으로 이 문제집으로 점수를 매깁니다.
- **베이스라인(1-5):**
  - **LLM 단독** — 문서 밖 세부 수치(예: 신생아 특례대출 2.5억, 소득대체율 38.6%)는 틀리거나 지어냅니다(환각).
  - **전체 문서 통째** — 잘 맞지만 매 질문마다 40페이지 전부를 보내 **느리고 비싸며**, 문서가 커지면 불가능합니다.
- 다음 세션부터 **필요한 부분만 검색**해서 전체 문서 통째와 같은 점수를 **싸고 빠르게** 내는 법(추출·검색·재정렬)을 배웁니다. 베이스라인이 앞으로 끌어올릴 출발선입니다.